In [1]:
from typing import Annotated
from langchain_core.tools import InjectedToolArg
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_groq import ChatGroq

e:\Lang_Chain\langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

In [ ]:
@tool
def get_conversion_factor(base_currency:str, target_currency:str)->float:
    """This function fetches currency factor for converting base currency into target currency"""
    url=f"https://v6.exchangerate-api.com/v6/{os.getenv("EXCHANGE_RATE_API")}/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()


In [5]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1781740801,
 'time_last_update_utc': 'Thu, 18 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1781827201,
 'time_next_update_utc': 'Fri, 19 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.6734}

In [ ]:
llm = ChatGroq(model="llama-3.3-70b-versatile")


In [7]:
llm_with_tool = llm.bind_tools([get_conversion_factor])

In [8]:
messages = [HumanMessage(content="Can you convert 20 USD into INR?")]
ai_message = llm_with_tool.invoke(messages)
messages.append(ai_message)

In [15]:
tool_result = get_conversion_factor.invoke(ai_message.tool_calls[0])
print(tool_result)

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1781740801, "time_last_update_utc": "Thu, 18 Jun 2026 00:00:01 +0000", "time_next_update_unix": 1781827201, "time_next_update_utc": "Fri, 19 Jun 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 94.6734}' name='get_conversion_factor' tool_call_id='d4yw0114v'


In [16]:

messages.append(tool_result)
result = llm_with_tool.invoke(messages)
print(result)

content='The conversion rate for 1 USD to INR is 94.6734. To convert 20 USD to INR, we multiply 20 by the conversion rate:\n\n20 * 94.6734 = 1893.468\n\nSo, 20 USD is approximately equal to 1893.47 INR.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 419, 'total_tokens': 486, 'completion_time': 0.175450363, 'completion_tokens_details': None, 'prompt_time': 0.021899023, 'prompt_tokens_details': None, 'queue_time': 0.051530137, 'total_time': 0.197349386}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019edbd6-56eb-7c31-8c78-f645d858159b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 419, 'output_tokens': 67, 'total_tokens': 486}
